In [2]:
# Import pakages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
import time
# Import Dask for processing large datasets
from dask import dataframe as dd
from scipy.stats import ks_2samp

## Load three datasets

In [3]:
# Product information
article = pd.read_csv("data/articles.csv")
article.head(1)

,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,...,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.


In [34]:
article.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105542 entries, 0 to 105541
Data columns (total 25 columns):
 #   Column                        Non-Null Count   Dtype 
---  ------                        --------------   ----- 
 0   article_id                    105542 non-null  int64 
 1   product_code                  105542 non-null  int64 
 2   prod_name                     105542 non-null  object
 3   product_type_no               105542 non-null  int64 
 4   product_type_name             105542 non-null  object
 5   product_group_name            105542 non-null  object
 6   graphical_appearance_no       105542 non-null  int64 
 7   graphical_appearance_name     105542 non-null  object
 8   colour_group_code             105542 non-null  int64 
 9   colour_group_name             105542 non-null  object
 10  perceived_colour_value_id     105542 non-null  int64 
 11  perceived_colour_value_name   105542 non-null  object
 12  perceived_colour_master_id    105542 non-null  int64 
 13 

In [23]:
# Customer information
customers = pd.read_csv("data/customers.csv/customers.csv")
# Drop the postal_code & FN, Active(unclear definitions) columns
customers.drop(columns= ["postal_code", "FN", "Active"], inplace = True)
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1371980 entries, 0 to 1371979
Data columns (total 4 columns):
 #   Column                  Non-Null Count    Dtype  
---  ------                  --------------    -----  
 0   customer_id             1371980 non-null  object 
 1   club_member_status      1365918 non-null  object 
 2   fashion_news_frequency  1355969 non-null  object 
 3   age                     1356119 non-null  float64
dtypes: float64(1), object(3)
memory usage: 41.9+ MB


----

## 1. Sampling large-scale retail data
### How can a representative sample be extracted from a large-scale retail transaction dataset while preserving key statistical properties of the full population?

- Check the original dataset descriptive statstics

In [5]:
# Transaction data 
transactions_train_dd = dd.read_csv('data/transactions_train.csv')

In [ ]:
# Numeric value
# Check std, mean
transactions_train_dd['price'].describe().compute() 

count    3.178832e+07
mean     2.782927e-02
std      1.918113e-02
min      1.694915e-05
25%      1.693220e-02
50%      3.015254e-02
75%      4.235593e-02
max      5.915254e-01
Name: price, dtype: float64

In [7]:
# Top frequency proportion of the categorical variable

article_proportions1 = transactions_train_dd['t_dat'].value_counts(normalize=True).compute()
article_proportions2 = transactions_train_dd['article_id'].value_counts(normalize=True).compute()
article_proportions3 = transactions_train_dd['sales_channel_id'].value_counts(normalize=True).compute()
article_proportions4 = transactions_train_dd['customer_id'].value_counts(normalize=True).compute()

print(article_proportions1.head(10))
print(article_proportions2.head(10))
print(article_proportions3.head(10))
print(article_proportions4.head(10))

t_dat
2019-09-28    0.006248
2020-04-11    0.005121
2019-11-29    0.005061
2018-11-23    0.004468
2018-09-29    0.004458
2019-07-24    0.003922
2019-04-06    0.003762
2020-06-17    0.003491
2019-11-28    0.003343
2019-06-19    0.003273
Name: proportion, dtype: double[pyarrow]
article_id
706016001    0.001582
706016002    0.001102
372860001    0.000998
610776002    0.000950
759871002    0.000828
464297007    0.000787
372860002    0.000769
610776001    0.000706
399223001    0.000700
706016003    0.000668
Name: proportion, dtype: float64
sales_channel_id
2    0.704028
1    0.295972
Name: proportion, dtype: float64
customer_id
be1981ab818cf4ef6765b2ecaea7a2cbf14ccd6e8a7ee985513d9e8e53c6d91b     0.00006
b4db5e5259234574edfff958e170fe3a5e13b6f146752ca066abca3c156acc71    0.000045
49beaacac0c7801c2ce2d189efe525fe80b5d37e46ed05b50a4cd88e34d0748f    0.000043
a65f77281a528bf5c1e9f270141d601d116e1df33bf9df512f495ee06647a9cc    0.000043
cd04ec2726dd58a8c753e0d6423e57716fd9ebcf2f14ed6012e7e5bea016b

- Sampling method 1 

In [8]:
tmp_list = []
# Always same index
np.random.seed(42)

# Because each Dask partition may contain a slightly different number of rows,
# we generate random indices based on an approximate partition size
sampled_idx = np.random.randint(0, int(len(transactions_train_dd) / (transactions_train_dd.npartitions + 1)), size=10000)

# Initialise the sampled DataFrame using the first partition
transactions_train_df = pd.DataFrame(transactions_train_dd.partitions[0]).loc[sampled_idx].sort_index()

# Iterate over remaining partitions and sample the same indices from each
for i in range(1, transactions_train_dd.npartitions):
    transactions_train_df = pd.concat([transactions_train_df, pd.DataFrame(transactions_train_dd.partitions[i]).loc[sampled_idx].sort_index()])

# Rename columns 
transactions_train_df.reset_index(drop=True, inplace=True)
transactions_train_df.rename(columns={0: 't_dat', 1: 'customer_id', 2: 'article_id', 3: 'price', 4: 'sales_channel_id'}, inplace=True)
transactions_train_df.tail()

,t_dat,customer_id,article_id,price,sales_channel_id
539995,2020-09-22,aaead1fa5369cd911dbec82a14bb279a3ae9523969cc32...,932798001,0.016932,2
539996,2020-09-22,ab9aead5b9d716f61c3071fa23c5528c1f5f72c8730b30...,861464001,0.016932,1
539997,2020-09-22,ac672405aa390e042f9a740a2e029c1ad0b143f3b69185...,904584007,0.025407,2
539998,2020-09-22,acb6effd34b902465c524bf62a170fc973baac58f619af...,909921001,0.025407,2
539999,2020-09-22,ad3663a848dccbddaf28127ccafa0b06c0f65408fc4d7b...,783517002,0.042034,2


In [ ]:
# The dataset of 31,788,324 records was split into 54 partitions, 
# from which 10,000 observations were sampled from each partition, resulting in a total of approximately 540,000 observations.

len(transactions_train_dd) # 31,788,324
len(transactions_train_df) # 540,000

540000

In [ ]:
# Each partition contains a similar number of observations, approximately between 588,400 and 589,100 rows.
partition_lengths = transactions_train_dd.map_partitions(len).compute()

print(partition_lengths[:5])

0    588429
1    588814
2    588812
3    588645
4    588552
dtype: int64


- Check the sample1 dataset descriptive statstics

In [10]:
# Numeric value
transactions_train_df['price'] = transactions_train_df['price'].apply(float)

In [11]:
transactions_train_df[['price']].describe()

,price
count,540000.000000
mean,0.027779
std,0.019240
min,0.000237
25%,0.015746
50%,0.025407
75%,0.033881
max,0.591525


In [12]:
# Top frequency proportion of the categorical variable
sample_proportions1 = transactions_train_df['t_dat'].value_counts(normalize=True)
sample_proportions2 = transactions_train_df['article_id'].value_counts(normalize=True)
sample_proportions3 = transactions_train_df['sales_channel_id'].value_counts(normalize=True)
sample_proportions4 = transactions_train_df['customer_id'].value_counts(normalize=True)
print(sample_proportions1.head(10))
print(sample_proportions2.head(10))
print(sample_proportions3.head(10))
print(sample_proportions4.head(10))

t_dat
2019-09-28    0.006259
2020-04-11    0.005174
2019-11-29    0.005091
2018-09-29    0.004570
2018-11-23    0.004259
2019-07-24    0.004144
2019-04-06    0.003811
2020-06-17    0.003648
2019-06-19    0.003343
2020-06-24    0.003302
Name: proportion, dtype: float64
article_id
706016001    0.001572
706016002    0.001085
372860001    0.000943
759871002    0.000926
610776002    0.000889
464297007    0.000798
372860002    0.000791
720125001    0.000680
399223001    0.000678
610776001    0.000676
Name: proportion, dtype: float64
sales_channel_id
2    0.703702
1    0.296298
Name: proportion, dtype: float64
customer_id
be1981ab818cf4ef6765b2ecaea7a2cbf14ccd6e8a7ee985513d9e8e53c6d91b    0.000065
c140410d72a41ee5e2e3ba3d7f5a860f337f1b5e41c27cf9bda5517c8774f8fa    0.000052
b4db5e5259234574edfff958e170fe3a5e13b6f146752ca066abca3c156acc71    0.000052
cd04ec2726dd58a8c753e0d6423e57716fd9ebcf2f14ed6012e7e5bea016b4d6    0.000048
a65f77281a528bf5c1e9f270141d601d116e1df33bf9df512f495ee06647a9cc    0

In [13]:
# Applied the Kolmogorov–Smirnov (K–S) test to compare the distributions of the original and sampled data
pop_price_series = transactions_train_dd['price'].compute()

ks_statistic, p_value = ks_2samp(
    transactions_train_df['price'], # Sample data
    pop_price_series)               # Population data 


# Small K-S statistic indicates minimal difference, but p-value < 0.05 suggests a statistically significant distribution shift
# Due to the large sample size, even a very small difference is detected as statistically significant.
print(f"K-S Statistic: {ks_statistic}")
print(f"P-value: {p_value}")

K-S Statistic: 0.001990176304985436
P-value: 0.029765721192088868


- sampling method 2

In [14]:
# A random sample of 1% of the entire dataset
sampled_df = transactions_train_dd.sample(frac=0.01, random_state=42)
sampled_df = sampled_df.compute()

In [15]:
sampled_df.head(5)

,t_dat,customer_id,article_id,price,sales_channel_id
111967,2018-09-22,e6d8e304aef4a2807ffae130a80fa1ee3315e56bbc568d...,683662002,0.013542,2
372951,2018-09-28,5e8f46a1250eeb4762049630a55b7359fbbb088ccd25c9...,630116011,0.032186,1
216683,2018-09-25,1c1a27a809eb94b3134593effa08afc9e8c200df10a4d1...,647982002,0.050831,1
519155,2018-09-29,bb9d48a93d7fdd78c17b759fe52cab553be83a4015e6f3...,666448006,0.020322,1
425105,2018-09-29,120ffd80762515ce84c8250e617ab98a97f623b0eee509...,633130005,0.013542,1


- Check the sample1 dataset descriptive statstics

In [16]:
sampled_df['price'].describe()

count    317880.000000
mean          0.027906
std           0.019283
min           0.000322
25%           0.015932
50%           0.025407
75%           0.033881
max           0.506780
Name: price, dtype: float64

In [17]:
sample_proportions1 = sampled_df['t_dat'].value_counts(normalize=True)
sample_proportions2 = sampled_df['article_id'].value_counts(normalize=True)
sample_proportions3 = sampled_df['sales_channel_id'].value_counts(normalize=True)
sample_proportions4 = sampled_df['customer_id'].value_counts(normalize=True)
print(sample_proportions1.head(10))
print(sample_proportions2.head(10))
print(sample_proportions3.head(10))
print(sample_proportions4.head(10))

t_dat
2019-09-28    0.006244
2019-11-29    0.005062
2020-04-11    0.005046
2018-11-23    0.004442
2018-09-29    0.004259
2019-07-24    0.003885
2019-04-06      0.0038
2020-06-17    0.003552
2019-11-28    0.003372
2019-06-19    0.003231
Name: proportion, dtype: double[pyarrow]
article_id
706016001    0.001696
372860001    0.001041
610776002    0.001022
706016002    0.001013
759871002    0.000853
372860002    0.000746
464297007    0.000739
156231001    0.000692
720125001    0.000673
562245046    0.000670
Name: proportion, dtype: float64
sales_channel_id
2    0.704565
1    0.295435
Name: proportion, dtype: float64
customer_id
b4db5e5259234574edfff958e170fe3a5e13b6f146752ca066abca3c156acc71     0.00006
898ede9fb639eb2aedb2d1d433eea958817e2bbb9f94524b3a0709af6fb5257e     0.00005
49beaacac0c7801c2ce2d189efe525fe80b5d37e46ed05b50a4cd88e34d0748f     0.00005
be1981ab818cf4ef6765b2ecaea7a2cbf14ccd6e8a7ee985513d9e8e53c6d91b     0.00005
851a88d56941fd5c43795a0f8bc1d2d04d66821e740930ae1aa6ca00f3a1b

In [18]:
# Applied the Kolmogorov–Smirnov (K–S)
ks_statistic, p_value = ks_2samp(
    sampled_df['price'],     # Sample data 
    pop_price_series         # Population data
)

print(f"K-S Statistic: {ks_statistic}")
print(f"P-value: {p_value}")

K-S Statistic: 0.002533006904283819
P-value: 0.03517985230525478


In [ ]:
# Final transaction sample data
transactions_train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 540000 entries, 0 to 539999
Data columns (total 5 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   t_dat             540000 non-null  datetime64[ns]
 1   customer_id       540000 non-null  object        
 2   article_id        540000 non-null  object        
 3   price             540000 non-null  float64       
 4   sales_channel_id  540000 non-null  object        
dtypes: datetime64[ns](1), float64(1), object(3)
memory usage: 20.6+ MB


-----

In [19]:
transactions_train_df['t_dat'] = pd.to_datetime(transactions_train_df['t_dat'])
transactions_train_df.head(1)

,t_dat,customer_id,article_id,price,sales_channel_id
0,2018-09-20,000aa7f0dc06cd7174389e76c9e132a67860c5f65f9706...,680912009,0.011847,2


In [ ]:
# Check for duplicates (5334 rows were removed)

transactions_train_df.drop_duplicates(inplace = True)
len(transactions_train_df)

534666

----

- Product data

In [ ]:
# Merge transaction_train_df & article data 
salesProduct = pd.merge(transactions_train_df, article, on='article_id', how='left')
salesProduct.head(1)
salesProduct[salesProduct.duplicated]

,t_dat,customer_id,article_id,price,sales_channel_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,...,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc


---

- Customer data 

In [ ]:
# Set unknown the club member status & news frequency
customers["club_member_status"].fillna("UNKNOWN", inplace=True)

customers["fashion_news_frequency"] = customers["fashion_news_frequency"].replace({"None":"NONE"})
customers["fashion_news_frequency"].fillna("UNKNOWN", inplace=True)

# Set missing values in age with the median
customers["age"].fillna(customers["age"].median(), inplace=True)

In [ ]:
#.to_csv("customers.csv", index = 0)

In [29]:
customers[customers.duplicated()]

,customer_id,club_member_status,fashion_news_frequency,age


In [ ]:
# Merge salesProduct & customers data
salesProductCustomer = pd.merge(salesProduct, customers, on='customer_id', how='left')
salesProductCustomer.head(2)

,t_dat,customer_id,article_id,price,sales_channel_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,...,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc,club_member_status,fashion_news_frequency,age
0,2018-09-20,000aa7f0dc06cd7174389e76c9e132a67860c5f65f9706...,680912009,0.011847,2,680912,Linni tee,255,T-shirt,Garment Upper body,...,2,Divided,53,Divided Collection,1005,Jersey Fancy,T-shirt in cotton jersey with a print motif on...,ACTIVE,NONE,22.0
1,2018-09-20,00a95aa4ba8d20f1bad415bab15ca6174a762f7550c819...,598795014,0.014390,1,598795,Dingo tee,255,T-shirt,Garment Upper body,...,2,Divided,53,Divided Collection,1005,Jersey Fancy,NaN,ACTIVE,NONE,28.0


In [31]:
salesProductCustomer[salesProductCustomer.duplicated()]

,t_dat,customer_id,article_id,price,sales_channel_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,...,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc,club_member_status,fashion_news_frequency,age


In [ ]:
# Save the final processed dataset
salesProductCustomer.to_csv("salesProductCustomer.csv", index = False)

In [ ]:
salesProductCustomer.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 534666 entries, 0 to 534665
Data columns (total 32 columns):
 #   Column                        Non-Null Count   Dtype         
---  ------                        --------------   -----         
 0   t_dat                         534666 non-null  datetime64[ns]
 1   customer_id                   534666 non-null  object        
 2   article_id                    534666 non-null  object        
 3   price                         534666 non-null  float64       
 4   sales_channel_id              534666 non-null  object        
 5   product_code                  534666 non-null  int64         
 6   prod_name                     534666 non-null  object        
 7   product_type_no               534666 non-null  int64         
 8   product_type_name             534666 non-null  object        
 9   product_group_name            534666 non-null  object        
 10  graphical_appearance_no       534666 non-null  int64         
 11  graphical_app